# 16 — Pull EPR (Ethnic Power Relations)

**Source:** Cederman, Wimmer & Min — Ethnic Power Relations (EPR-Core) dataset  
**Provider:** International Conflict Research group, ETH Zurich  
**Coverage:** Global, 1946–2020, annual at the ethnic-group level  
**Access:** Direct CSV download from icr.ethz.ch (no registration required)

## What this notebook does

1. Downloads the EPR-Core CSV from ETH Zurich
2. Aggregates from ethnic-group-year to country-year, deriving the variables
   used across the instability prediction literature
3. Writes a country-year parquet to ADLS

## Why EPR is needed

EPR provides the ethnic group exclusion variables that are the primary predictors
in ethnic war and genocide models (Goldstone 2010, Harff 2003, Goldsmith 2013)
and a key predictor in the ViEWS civil conflict model. The synthesisdocument
identifies `ethnic_exclusion` (large group politically excluded) as a strong
predictor that is absent from all other sources currently being acquired.

The raw EPR data is at the **ethnic group × year** level. This notebook aggregates
it to **country × year** by computing group-level flags and summing or flagging
across groups within each country-year.

## Variables produced

| Variable | Description | Use |
|---|---|---|
| `ethnic_exclusion_any` | 1 if any group >10% of population has status Powerless or Discriminated | **Dependent variable** (ethnic war models) + predictor |
| `n_excluded_groups` | Count of politically excluded groups (any size) | Predictor |
| `excluded_pop_share` | Sum of population shares of excluded groups | Predictor |
| `n_relevant_groups` | Count of all politically relevant ethnic groups | Predictor (fractionalization proxy) |
| `max_group_size` | Population share of the largest single group | Predictor |
| `dominant_group_flag` | 1 if any group holds Monopoly or Dominant status | Predictor (elite concentration) |
| `epr_state_collapse` | 1 if EPR codes the country as State Collapse this year | **Dependent variable** |

## EPR status codes

| Status | Meaning | Excluded? |
|---|---|---|
| Monopoly | Group rules alone | No |
| Dominant | Group dominates; others in junior role | No |
| Senior Partner | Shares power as senior partner | No |
| Junior Partner | Shares power as junior partner | No |
| Powerless | No access to central power | **Yes** |
| Discriminated | Actively discriminated against by state | **Yes** |
| Self-exclusion | Group self-excludes from politics | No |
| Irrelevant | Not politically mobilised | No |
| State Collapse | No functioning central authority | — |

## Output path on ADLS

```
raw/epr/{RUN_DATE}/epr_annual.parquet
```

Notebook `02/02_build_feature_matrix` already includes `"epr": "raw/epr"` in `RAW_PREFIXES`.

In [ ]:
%%capture
%pip install requests pandas pyarrow azure-identity adlfs --quiet

In [ ]:
import io
import os
import warnings
from datetime import datetime
from pathlib import Path

import pandas as pd
import numpy as np
import requests
import adlfs
from azure.identity import DefaultAzureCredential

warnings.filterwarnings('ignore')
RUN_DATE = datetime.utcnow().strftime('%Y%m%d')

In [ ]:
ADLS_ACCOUNT_NAME = os.environ['ADLS_ACCOUNT_NAME']
ADLS_CONTAINER    = os.getenv('ADLS_CONTAINER', 'data')
CROSSWALK_PATH    = '../data/country_crosswalk.csv'

# EPR-Core 2021 CSV — direct download from ETH Zurich ICR group
# If the URL has been updated, check https://icr.ethz.ch/data/epr/core/
EPR_URL = 'https://icr.ethz.ch/data/epr/core/EPR-2021.csv'
EPR_URL_FALLBACK = 'https://icr.ethz.ch/data/epr/core/EPR-2019.csv'

# Groups are 'large' if they account for more than this share of population
LARGE_GROUP_THRESHOLD = 0.10

# Statuses that constitute political exclusion
EXCLUDED_STATUSES = {'powerless', 'discriminated'}

In [ ]:
credential = DefaultAzureCredential()
storage_options = {'account_name': ADLS_ACCOUNT_NAME, 'credential': credential}

def adls_path(subpath):
    return f'abfss://{ADLS_CONTAINER}@{ADLS_ACCOUNT_NAME}.dfs.core.windows.net/{subpath}'

def write_parquet(df, subpath):
    df.to_parquet(adls_path(subpath), storage_options=storage_options,
                  index=False, engine='pyarrow')
    print(f'  Written {len(df):,} rows → {subpath}')

In [ ]:
# Download EPR-Core
df_epr_raw = None
for url in [EPR_URL, EPR_URL_FALLBACK]:
    try:
        print(f'Downloading EPR from {url} ...')
        resp = requests.get(url, timeout=60, headers={'User-Agent': 'Mozilla/5.0'})
        resp.raise_for_status()
        df_epr_raw = pd.read_csv(io.BytesIO(resp.content), low_memory=False)
        print(f'  Raw shape: {df_epr_raw.shape}')
        print(f'  Columns  : {df_epr_raw.columns.tolist()}')
        break
    except Exception as exc:
        print(f'  Failed: {exc}')

if df_epr_raw is None:
    raise RuntimeError(
        'Could not download EPR data. '
        'Download manually from https://icr.ethz.ch/data/epr/core/ '
        'and load with: df_epr_raw = pd.read_csv("EPR-2021.csv")'
    )

In [ ]:
# Standardise column names — EPR column names are stable across versions
df = df_epr_raw.copy()
df.columns = [c.strip().lower() for c in df.columns]

# Core columns in EPR-Core:
#   gwgroupid, statename, gwid (G&W country code), year,
#   group, groupid, size, status, reg_aut, umbrella
print('Columns:', df.columns.tolist())
print(f'Rows: {len(df):,}')
print(f'Years: {df["year"].min()}–{df["year"].max()}')
print('Status values:', df['status'].unique().tolist())

In [ ]:
# Map Gleditsch-Ward country code (gwid) to ISO3 via crosswalk
cw_path = Path(CROSSWALK_PATH)
if not cw_path.exists():
    cw_path = Path('data/country_crosswalk.csv')

df_cw = pd.read_csv(cw_path, dtype=str)
df_cw['gw_numeric'] = pd.to_numeric(df_cw['gw_numeric'], errors='coerce')
gw_to_iso3 = dict(zip(df_cw['gw_numeric'], df_cw['iso3']))

df['iso3'] = pd.to_numeric(df['gwid'], errors='coerce').map(gw_to_iso3)

unmatched_gw = df.loc[df['iso3'].isna(), 'gwid'].unique()
print(f'Unmatched GW codes ({len(unmatched_gw)}): {sorted(unmatched_gw)[:10]}')

df = df.dropna(subset=['iso3', 'year', 'size', 'status'])
df['year']   = df['year'].astype(int)
df['size']   = pd.to_numeric(df['size'], errors='coerce').fillna(0.0)
df['status'] = df['status'].str.strip().str.lower()

In [ ]:
# Aggregate from group-year to country-year
def _agg_country_year(grp):
    statuses = grp['status'].values
    sizes    = grp['size'].values

    excluded_mask      = np.isin(statuses, list(EXCLUDED_STATUSES))
    large_mask         = sizes >= LARGE_GROUP_THRESHOLD
    state_collapse     = int(any(s == 'state collapse' for s in statuses))
    dominant_statuses  = {'monopoly', 'dominant'}

    return pd.Series({
        # Primary dependent variable: any large group excluded
        'ethnic_exclusion_any': int(
            any(e and l for e, l in zip(excluded_mask, large_mask))
        ),
        # Finer-grained predictors
        'n_excluded_groups':    int(excluded_mask.sum()),
        'excluded_pop_share':   float(sizes[excluded_mask].sum()),
        'n_relevant_groups':    len(grp),
        'max_group_size':       float(sizes.max()) if len(sizes) else 0.0,
        'dominant_group_flag':  int(
            any(s in dominant_statuses for s in statuses)
        ),
        'epr_state_collapse':   state_collapse,
    })


df_annual = (
    df.groupby(['iso3', 'year'])
    .apply(_agg_country_year)
    .reset_index()
)

print(f'Country-year panel: {len(df_annual):,} rows')
print(f'Countries         : {df_annual["iso3"].nunique()}')
print(f'Years             : {df_annual["year"].min()}–{df_annual["year"].max()}')
print()
excl_rate = df_annual['ethnic_exclusion_any'].mean()
print(f'ethnic_exclusion_any base rate : {excl_rate:.1%}')
print(f'epr_state_collapse base rate   : {df_annual["epr_state_collapse"].mean():.1%}')
df_annual.head()

In [ ]:
write_parquet(df_annual, f'raw/epr/{RUN_DATE}/epr_annual.parquet')

print()
print('=' * 55)
print('Notebook 14e complete')
print('=' * 55)
print(f'  Rows      : {len(df_annual):,}')
print(f'  Countries : {df_annual["iso3"].nunique()}')
print(f'  Years     : {df_annual["year"].min()}–{df_annual["year"].max()}')
print()
print('Dependent variables written:')
print('  ethnic_exclusion_any  — large group (>10%) Powerless or Discriminated')
print('  epr_state_collapse    — EPR codes country as state collapse')
print()
print('Next: add  "epr": "raw/epr"  to RAW_PREFIXES in notebook 14.')